# Unidade 3 · Aprendizagem não supervisionada
**Aprendizagem Computacional · 4º TIA · Apostila do Aluno · versão INICIAL**

Este notebook cobre os cap. 18 a 22 e 24 da apostila. Vamos agrupar **178 vinhos** de uma região da Itália a partir de 13 medidas químicas, **sem usar o rótulo**, com os cinco tipos de agrupamento da ementa:

| Seção | Método | Capítulo |
|---|---|---|
| 2 | Avaliação de agrupamentos | 19 |
| 3 | Partição: K-Means (+ hierárquico e DBSCAN como contraste) | 20 |
| 4 | Probabilístico: mistura de gaussianas | 21 |
| 5 | Difuso: Fuzzy C-Means, implementado do zero | 22 |
| 6 | Mapa auto-organizável (SOM), implementado do zero | 23 |
| 7 | Do grupo à decisão: perfis | 24 |

O conjunto tem um rótulo verdadeiro (3 cultivares). Ele fica **guardado** e só aparece no fim, como checagem, porque na vida real o agrupamento é usado justamente quando não existe rótulo.

**Como usar a versão INICIAL:** procure os `# TODO`.

## 1 · Dados e padronização

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_wine
from sklearn.preprocessing import StandardScaler
SEMENTE = 42

vinhos = load_wine(as_frame=True)
X = vinhos.data
rotulo_guardado = vinhos.target          # NÃO usar até a seção 7
Z = None   # TODO: padronize X (cap. 7). Por que isso é obrigatório aqui?
print(X.shape); X.describe().T[['mean', 'std', 'min', 'max']].round(2)

**Leitura:** a prolina varia de centenas a mais de mil, enquanto outras medidas variam em décimos. Sem padronizar, a prolina decidiria sozinha quem é parecido com quem.

## 2 · Como saber se um agrupamento é bom (cap. 18)
Sem rótulo, avaliamos a **coesão** (pontos perto do seu grupo) e a **separação** (grupos longe uns dos outros):
- **Inércia** (soma das distâncias² ao centro): sempre cai quando k aumenta, então procure o \"cotovelo\".
- **Silhueta** (−1 a 1): maior é melhor.
- **Davies-Bouldin**: menor é melhor.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score

linhas = []
for k in range(2, 9):
    km = KMeans(n_clusters=k, n_init=10, random_state=SEMENTE).fit(Z)
    linhas.append({'k': k, 'inércia': km.inertia_, 'silhueta': silhouette_score(Z, km.labels_),
                   'Davies-Bouldin': davies_bouldin_score(Z, km.labels_)})
avaliacao = pd.DataFrame(linhas).set_index('k')
fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))
avaliacao['inércia'].plot(ax=ax[0], marker='o', title='Cotovelo (inércia)')
avaliacao['silhueta'].plot(ax=ax[1], marker='o', title='Silhueta média')
plt.tight_layout(); plt.show()
avaliacao.round(3)

**Leitura:** o cotovelo aparece em k = 3, e a silhueta também é máxima em k = 3. Três critérios independentes concordam. Quando discordarem, a decisão final é do especialista do domínio: quantos grupos o gestor consegue usar?

## 3 · Agrupamento por partição: K-Means (cap. 19)

In [ ]:
from sklearn.decomposition import PCA
# TODO: K-Means com 3 grupos, n_init=10, random_state=SEMENTE, ajustado em Z
km = ...
grupos_km = km.labels_
P = PCA(n_components=2).fit(Z)
Z2 = P.transform(Z)
plt.scatter(Z2[:, 0], Z2[:, 1], c=grupos_km, cmap='viridis', s=18)
centros = P.transform(km.cluster_centers_)
plt.scatter(centros[:, 0], centros[:, 1], marker='X', s=200, c='red', label='centróides')
plt.xlabel('PC1'); plt.ylabel('PC2'); plt.legend(); plt.title('K-Means (k = 3) visto no plano do PCA'); plt.show()
print('Vinhos por grupo:', np.bincount(grupos_km))

### Contraste: hierárquico e DBSCAN

In [ ]:
from scipy.cluster.hierarchy import linkage, dendrogram
from sklearn.cluster import AgglomerativeClustering, DBSCAN

plt.figure(figsize=(10, 3.2))
dendrogram(linkage(Z, method='ward'), no_labels=True, color_threshold=20)
plt.title('Dendrograma (Ward): cortar na altura que gera 3 ramos'); plt.ylabel('distância de fusão'); plt.show()
grupos_hier = AgglomerativeClustering(n_clusters=3, linkage='ward').fit_predict(Z)

grupos_db = DBSCAN(eps=2.3, min_samples=5).fit_predict(Z)
print('DBSCAN · grupos:', sorted(set(grupos_db) - {-1}), '| pontos marcados como ruído (-1):', (grupos_db == -1).sum())

**Leitura:** o dendrograma mostra três ramos bem separados, coerente com o K-Means. O DBSCAN, que procura regiões densas, sofre com 13 dimensões (cap. 7, maldição da dimensionalidade) e marca muitos vinhos como ruído. Ele brilha em poucas dimensões, como coordenadas geográficas de focos de queimada.

## 4 · Agrupamento probabilístico: mistura de gaussianas (cap. 20)

In [ ]:
from sklearn.mixture import GaussianMixture
bics = {k: GaussianMixture(n_components=k, covariance_type='diag', random_state=SEMENTE).fit(Z).bic(Z)
        for k in range(1, 7)}
print('BIC por número de componentes (menor é melhor):', {k: round(v) for k, v in bics.items()})
gmm = GaussianMixture(n_components=3, covariance_type='diag', random_state=SEMENTE).fit(Z)
proba = None   # TODO: probabilidade de cada vinho pertencer a cada componente (método predict_proba)
grupos_gmm = proba.argmax(axis=1)
certeza = proba.max(axis=1)
print('Vinhos com pertinência máxima < 0,9 (casos de fronteira):', (certeza < 0.9).sum())
pd.DataFrame(proba[np.argsort(certeza)[:5]], columns=['P(grupo 0)', 'P(grupo 1)', 'P(grupo 2)']).round(3)

**Leitura:** diferente do K-Means, a mistura de gaussianas diz **quão certo** está cada vinho em seu grupo. Os casos de fronteira são justamente os que um especialista deveria olhar.

## 5 · Agrupamento difuso: Fuzzy C-Means (cap. 21)
O scikit-learn não tem Fuzzy C-Means, então vamos implementá-lo. São poucas linhas, e ver o algoritmo por dentro ajuda a entendê-lo. Cada vinho recebe um **grau de pertinência** entre 0 e 1 em cada grupo, e os graus de um vinho somam 1. O parâmetro *m* > 1 controla o quanto as fronteiras são \"borradas\".

In [ ]:
def fuzzy_c_means(Z, c=3, m=2.0, max_iter=300, tol=1e-6, semente=42):
    rng = np.random.default_rng(semente)
    U = rng.random((len(Z), c)); U /= U.sum(axis=1, keepdims=True)      # pertinências iniciais
    for _ in range(max_iter):
        Um = U ** m
        centros = (Um.T @ Z) / Um.sum(axis=0)[:, None]                    # média ponderada pelas pertinências
        d = np.linalg.norm(Z[:, None, :] - centros[None, :, :], axis=2) + 1e-12
        novo = 1.0 / ((d[:, :, None] / d[:, None, :]) ** (2 / (m - 1))).sum(axis=2)
        if np.abs(novo - U).max() < tol:
            U = novo; break
        U = novo
    return U, centros

U, centros_fcm = fuzzy_c_means(Z, c=3, m=2.0)
grupos_fcm = U.argmax(axis=1)
coef_particao = (U ** 2).sum() / len(Z)    # 1 = partição nítida; 1/c = totalmente difusa
print(f'Coeficiente de partição: {coef_particao:.3f}  (mínimo possível com 3 grupos: {1/3:.3f})')
print('Vinhos por grupo (maior pertinência):', np.bincount(grupos_fcm))
pd.DataFrame(U[:5], columns=['μ grupo 0', 'μ grupo 1', 'μ grupo 2']).round(3)

**Pergunta para discussão:** qual a diferença entre a pertinência do Fuzzy C-Means e a probabilidade da mistura de gaussianas?

**Resposta esperada:** na prática, as duas dão números entre 0 e 1 que somam 1. Mas a probabilidade vem de um modelo estatístico (\"qual componente gerou este ponto?\") e a pertinência é um grau de semelhança, vindo da lógica difusa (\"o quanto este ponto é do grupo A\"). Um bairro de transição entre área residencial e comercial é parcialmente dos dois; não há \"um gerador verdadeiro\".

## 6 · Mapa auto-organizável, SOM (cap. 22)
Um SOM é uma grade de neurônios (aqui, 6 × 6). Cada neurônio tem um vetor de pesos do tamanho de um vinho. A cada passo, o neurônio mais parecido com um vinho sorteado (o **vencedor**) e seus **vizinhos na grade** se aproximam desse vinho. No fim, vinhos parecidos caem em neurônios vizinhos, e a grade vira um **mapa 2D** dos dados.

In [ ]:
def treinar_som(Z, linhas=6, colunas=6, iteracoes=3000, taxa0=0.5, raio0=3.0, semente=42):
    rng = np.random.default_rng(semente)
    pesos = rng.normal(size=(linhas, colunas, Z.shape[1])) * 0.1
    grade = np.stack(np.meshgrid(np.arange(linhas), np.arange(colunas), indexing='ij'), axis=-1)
    for t in range(iteracoes):
        frac = t / iteracoes
        taxa, raio = taxa0 * (1 - frac), raio0 * (1 - frac) + 0.5          # ambos encolhem com o tempo
        x = Z[rng.integers(len(Z))]
        dist = np.linalg.norm(pesos - x, axis=2)
        vencedor = np.array(np.unravel_index(dist.argmin(), dist.shape))
        h = np.exp(-((grade - vencedor) ** 2).sum(axis=2) / (2 * raio ** 2))   # vizinhança gaussiana
        pesos += taxa * h[:, :, None] * (x - pesos)
    return pesos

def neuronio_vencedor(pesos, Z):
    d = np.linalg.norm(pesos[None, :, :, :] - Z[:, None, None, :], axis=3)
    idx = d.reshape(len(Z), -1).argmin(axis=1)
    return np.stack(np.unravel_index(idx, pesos.shape[:2]), axis=1)

pesos = treinar_som(Z)
posicoes = neuronio_vencedor(pesos, Z)

# Matriz-U: distância média de cada neurônio aos vizinhos. Valores altos = fronteiras entre grupos.
L, C, _ = pesos.shape
umat = np.zeros((L, C))
for i in range(L):
    for j in range(C):
        viz = [pesos[a, b] for a, b in [(i-1, j), (i+1, j), (i, j-1), (i, j+1)] if 0 <= a < L and 0 <= b < C]
        umat[i, j] = np.mean([np.linalg.norm(pesos[i, j] - v) for v in viz])

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
im = ax[0].imshow(umat, cmap='Greys'); ax[0].set_title('Matriz-U (claro = parecido, escuro = fronteira)')
plt.colorbar(im, ax=ax[0])
rng = np.random.default_rng(SEMENTE)
jit = rng.uniform(-0.3, 0.3, size=posicoes.shape)
ax[1].scatter(posicoes[:, 1] + jit[:, 1], posicoes[:, 0] + jit[:, 0], c=grupos_km, cmap='viridis', s=16)
ax[1].set_xlim(-0.5, C - 0.5); ax[1].set_ylim(L - 0.5, -0.5); ax[1].set_aspect('equal')
ax[1].set_title('Onde cada vinho caiu (cor = grupo do K-Means)')
plt.tight_layout(); plt.show()

**Leitura:** as cores do K-Means ocupam regiões contíguas do mapa, e a matriz-U mostra faixas escuras (fronteiras) entre elas. O SOM chegou a uma organização compatível com o K-Means por um caminho totalmente diferente. Como o SOM preserva a vizinhança, ele também mostra **quais grupos são vizinhos**, algo que o K-Means não diz.

## 7 · Checagem com o rótulo guardado e perfis dos grupos (cap. 18 e 24)
Agora, e só agora, comparamos com as cultivares verdadeiras pelo **ARI** (*Adjusted Rand Index*: 1 = mesma partição, 0 = acaso). Também comparamos os métodos entre si.

In [ ]:
from sklearn.metrics import adjusted_rand_score
particoes = {'K-Means': grupos_km, 'Hierárquico (Ward)': grupos_hier, 'Mistura gaussiana': grupos_gmm,
             'Fuzzy C-Means': grupos_fcm}
pd.DataFrame({'ARI vs. cultivar verdadeira': {n: adjusted_rand_score(rotulo_guardado, g) for n, g in particoes.items()},
              'ARI vs. K-Means': {n: adjusted_rand_score(grupos_km, g) for n, g in particoes.items()}}).round(3)

In [ ]:
perfil = X.assign(grupo=grupos_km).groupby('grupo')[['alcohol', 'flavanoids', 'color_intensity', 'proline']].mean()
perfil['vinhos'] = np.bincount(grupos_km)
perfil.round(2)

**Do grupo à decisão (cap. 24):** um número de grupo não diz nada a ninguém. O passo final é a **tabela de perfil** em unidades originais, que vira um nome que o usuário entende: \"alto teor alcoólico e muita prolina\", \"pouco flavonoide e cor intensa\". No seu projeto, esses nomes são o que vai para o dashboard.

## Checklist da Unidade 3
- [ ] Padronizei antes de agrupar.
- [ ] Escolhi o número de grupos com pelo menos 2 critérios (cotovelo, silhueta, BIC) e o bom senso do domínio.
- [ ] Comparei pelo menos 2 métodos e vi se concordam.
- [ ] Usei pertinência ou probabilidade para achar os casos de fronteira.
- [ ] Descrevi cada grupo em unidades originais e dei nomes compreensíveis.
- [ ] Não tratei os grupos como \"classes verdadeiras\".